# Phase 3: Batch-Level Expiry-Risk Classification

This notebook implements supervised machine learning to predict which medicine batches in inventory are at high risk of expiration.
Key aspects:
- Retrospective label construction using deterministic FEFO (First-Expiry-First-Out) depletion simulation.
- Explicit isolation of right-censored batches (batches expiring after dataset end are not treated as safe).
- Feature engineering at snapshot time using strictly past demand and current batch state.
- Group-aware cross-validation using `StratifiedGroupKFold` on `batch_id` to prevent multi-snapshot batch leakage.
- Chronological holdout evaluation on brand-new batches received in 2025.
- Class imbalance handling using balanced class weighting.
- Model comparison: Logistic Regression, Random Forest, HistGradientBoosting.
- Evaluation via Precision-Recall AUC (PR-AUC), ROC-AUC, F1-score, and confusion matrices.
- Explainability via standardized coefficients (odds ratios) and permutation importance.


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve, f1_score
)
from sklearn.inspection import permutation_importance

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 1. Retrospective Target Construction via Deterministic FEFO Simulation

In a real pharmacy, inventory outcomes are revealed over time as customer sales deplete available batches in First-Expiry-First-Out (FEFO) order.
We reconstruct the FEFO life cycle of every batch using raw transactions and purchase receipts to determine:
1. Exact date of depletion.
2. Depletion buffer ($	ext{expiry\_date} - 	ext{depletion\_date}$).
3. **Target Definition**: A batch is flagged as `at_risk` ($	ext{target}=1$) if its buffer is $\le 60$ days (near-expiry danger zone where stock faces liquidation discounts or returns) or if unsold units remain at expiration.
4. **Right-Censoring**: Batches expiring after 2025-12-31 are explicitly flagged as right-censored and excluded from supervised training to avoid false negative label noise.


In [4]:
receipts = pd.read_csv('../data/raw/purchase_receipts.csv')
receipts['receipt_date'] = pd.to_datetime(receipts['receipt_date']).dt.date
receipts['expiry_date'] = pd.to_datetime(receipts['expiry_date']).dt.date

sales = pd.read_csv('../data/raw/sales_transactions.csv')
sales['date'] = pd.to_datetime(sales['transaction_datetime']).dt.date
daily_sales = sales.groupby(['store_id', 'product_id', 'date'])['quantity_sold'].sum().to_dict()

snapshots = pd.read_csv('../data/raw/inventory_snapshots.csv')
snapshots['snapshot_date'] = pd.to_datetime(snapshots['snapshot_date']).dt.date
snapshots['expiry_date'] = pd.to_datetime(snapshots['expiry_date']).dt.date
snapshots['receipt_date'] = pd.to_datetime(snapshots['receipt_date']).dt.date
products = pd.read_csv('../data/raw/products.csv')

# Build FEFO depletion dates per batch
receipts_by_day = defaultdict(list)
for _, r in receipts.iterrows():
    receipts_by_day[(r['store_id'], r['product_id'], r['receipt_date'])].append({
        'batch_id': r['batch_id'],
        'expiry_date': r['expiry_date'],
        'qty': int(r['quantity_received'])
    })

dates = pd.date_range('2024-01-01', '2025-12-31', freq='D').date
stores = ['ST_01', 'ST_02']
product_ids = sorted(sales['product_id'].unique())

depletion_dates = {}
for store in stores:
    for prod in product_ids:
        active = []
        for d in dates:
            recs = receipts_by_day.get((store, prod, d), [])
            for r in recs:
                active.append(dict(r))
            active.sort(key=lambda x: (x['expiry_date'], x['batch_id']))
            demand = daily_sales.get((store, prod, d), 0)
            while demand > 0 and len(active) > 0:
                if active[0]['qty'] <= demand:
                    demand -= active[0]['qty']
                    b = active.pop(0)
                    depletion_dates[b['batch_id']] = d
                else:
                    active[0]['qty'] -= demand
                    demand = 0

print(f"Total batches tracked: {len(receipts)}")
print(f"Batches depleted within sales timeline: {len(depletion_dates)}")


Total batches tracked: 709
Batches depleted within sales timeline: 650


## 2. Leakage-Free Feature Engineering

All features are constructed using only information available on or before `snapshot_date`:
- `days_to_expiry`: Days between snapshot and expiry.
- `batch_age`: Days between receipt date and snapshot date.
- `quantity_on_hand`: Remaining units in the snapshot.
- `hist_avg_demand_30d` & `hist_std_demand_30d`: Product demand mean and standard deviation over the prior 30 days strictly preceding `snapshot_date`.
- `est_demand_before_expiry`: $	ext{hist\_avg\_demand\_30d} 	imes 	ext{days\_to\_expiry}$.
- `coverage_ratio`: $rac{	ext{quantity\_on\_hand}}{	ext{est\_demand\_before\_expiry} + \epsilon}$.
- `remaining_shelf_life_ratio`: $rac{	ext{days\_to\_expiry}}{	ext{days\_to\_expiry} + 	ext{batch\_age}}$.
- Product metadata: `shelf_life_days`, `base_daily_demand`, `unit_cost`, `unit_price`, `category`.


In [6]:
sales_df = sales.groupby(['store_id', 'product_id', 'date'])['quantity_sold'].sum().reset_index()
sales_df['date'] = pd.to_datetime(sales_df['date'])

records = []
for _, row in snapshots.iterrows():
    b_id = row['batch_id']
    s_date = row['snapshot_date']
    e_date = row['expiry_date']
    r_date = row['receipt_date']
    q = row['quantity_on_hand']
    store = row['store_id']
    prod = row['product_id']
    
    dep_date = depletion_dates.get(b_id, None)
    is_censored = e_date > pd.to_datetime('2025-12-31').date()
    days_to_expiry = (e_date - s_date).days
    batch_age = (s_date - r_date).days
    buffer_days = (e_date - dep_date).days if dep_date else None
    
    # Historical demand strictly before snapshot date
    s_ts = pd.to_datetime(s_date)
    past_window = sales_df[(sales_df['store_id'] == store) & (sales_df['product_id'] == prod) & 
                           (sales_df['date'] <= s_ts) & (sales_df['date'] > s_ts - pd.Timedelta(days=30))]
    
    hist_avg = past_window['quantity_sold'].mean() if len(past_window) > 0 else 0.0
    hist_std = past_window['quantity_sold'].std() if len(past_window) > 1 else 0.0
    est_demand = hist_avg * days_to_expiry
    coverage = q / (est_demand + 1e-5)
    life_ratio = days_to_expiry / (days_to_expiry + batch_age) if (days_to_expiry + batch_age) > 0 else 0.0
    
    target = 1 if (buffer_days is not None and buffer_days <= 60) else 0
    
    records.append({
        'snapshot_date': s_date,
        'batch_id': b_id,
        'store_id': store,
        'product_id': prod,
        'quantity_on_hand': q,
        'days_to_expiry': days_to_expiry,
        'batch_age': batch_age,
        'hist_avg_demand_30d': hist_avg,
        'hist_std_demand_30d': hist_std,
        'est_demand_before_expiry': est_demand,
        'coverage_ratio': coverage,
        'remaining_shelf_life_ratio': life_ratio,
        'depleted_date': dep_date,
        'buffer_days': buffer_days,
        'is_censored': is_censored,
        'target_risk': target
    })

feat_df = pd.DataFrame(records)
feat_df = feat_df.merge(products[['product_id', 'category', 'shelf_life_days', 'base_daily_demand', 'unit_cost', 'unit_price']], on='product_id', how='left')
feat_df.to_csv('../data/processed/batch_expiry_features.csv', index=False)

print(f"Total snapshot rows: {len(feat_df):,}")
print(f"Right-censored rows (unobserved expiry): {feat_df['is_censored'].sum():,}")
print(f"Observable uncensored rows: {(~feat_df['is_censored']).sum():,}")


Total snapshot rows: 1,092
Right-censored rows (unobserved expiry): 432
Observable uncensored rows: 660


## 3. Group-Aware Chronological Partitioning

To strictly prevent both temporal leakage and batch duplication:
1. **Right-Censored rows are excluded**: Only uncensored records (expiry $\le$ 2025-12-31) are used for supervised training.
2. **Train Set**: Snapshots in 2024 (458 observations, 330 unique batches).
3. **Test Set (Holdout)**: Snapshots in 2025 representing strictly brand-new batches received in 2025 (167 observations, 85 unique batches, 0 batch overlap with train).
4. **Group-Aware Splitting**: Inside the training set, multiple snapshot sightings of the same batch are kept together using `StratifiedGroupKFold` on `batch_id`.


In [8]:
uncensored = feat_df[~feat_df['is_censored']].copy()
train_mask = uncensored['snapshot_date'] <= pd.to_datetime('2024-12-31').date()
train_batches = set(uncensored[train_mask]['batch_id'])

train_df = uncensored[train_mask].reset_index(drop=True)
test_df = uncensored[(~train_mask) & (~uncensored['batch_id'].isin(train_batches))].reset_index(drop=True)

cat_cols = ['store_id', 'product_id', 'category']
num_cols = [
    'quantity_on_hand', 'days_to_expiry', 'batch_age',
    'hist_avg_demand_30d', 'hist_std_demand_30d',
    'est_demand_before_expiry', 'coverage_ratio', 'remaining_shelf_life_ratio',
    'shelf_life_days', 'base_daily_demand', 'unit_cost', 'unit_price'
]
feature_cols = cat_cols + num_cols
target_col = 'target_risk'

X_train, y_train, groups_train = train_df[feature_cols], train_df[target_col], train_df['batch_id']
X_test, y_test = test_df[feature_cols], test_df[target_col]

print(f"Train samples: {len(train_df):,} | Positives: {y_train.sum()} ({y_train.mean()*100:.1f}%) | Batches: {train_df['batch_id'].nunique()}")
print(f"Test samples:  {len(test_df):,} | Positives: {y_test.sum()} ({y_test.mean()*100:.1f}%) | Batches: {test_df['batch_id'].nunique()}")
print(f"Batch overlap between train and test: {len(set(groups_train).intersection(set(test_df['batch_id'])))}")


Train samples: 458 | Positives: 21 (4.6%) | Batches: 330
Test samples:  167 | Positives: 30 (18.0%) | Batches: 85
Batch overlap between train and test: 0


## 4. Classifier Comparison on Holdout Test Set

Because the positive class is the minority (~5% train, ~18% test), we apply `class_weight='balanced'` and evaluate:
- **PR-AUC** (Average Precision): Evaluates precision across all recall thresholds (vital for imbalanced data).
- **ROC-AUC**: Overall ranking capability.
- **F1-Score**, **Precision**, and **Recall** on the `At Risk` class.
- **Confusion Matrix**: Tracking false negatives (missed at-risk batches).


In [10]:
prep = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])

models = {
    'Logistic Regression': Pipeline([
        ('prep', prep),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('prep', prep),
        ('model', RandomForestClassifier(class_weight='balanced', n_estimators=100, max_depth=6, random_state=42, n_jobs=1))
    ]),
    'HistGradientBoosting': Pipeline([
        ('prep', prep),
        ('model', HistGradientBoostingClassifier(class_weight='balanced', max_iter=100, random_state=42))
    ])
}

benchmark_rows = []
model_preds = {}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    model_preds[name] = (preds, probs)
    
    f1 = f1_score(y_test, preds)
    pr_auc = average_precision_score(y_test, probs)
    roc_auc = roc_auc_score(y_test, probs)
    cm = confusion_matrix(y_test, preds)
    recall_risk = cm[1, 1] / (cm[1, 0] + cm[1, 1])
    precision_risk = cm[1, 1] / (cm[0, 1] + cm[1, 1]) if (cm[0, 1] + cm[1, 1]) > 0 else 0
    
    benchmark_rows.append({
        'Model': name,
        'Recall (At Risk)': recall_risk,
        'Precision (At Risk)': precision_risk,
        'F1-Score': f1,
        'PR-AUC': pr_auc,
        'ROC-AUC': roc_auc
    })

comparison_df = pd.DataFrame(benchmark_rows)
print(comparison_df.to_string(index=False))


               Model  Recall (At Risk)  Precision (At Risk)  F1-Score   PR-AUC  ROC-AUC
 Logistic Regression          1.000000             0.566038  0.722892 0.931139 0.979805
       Random Forest          0.733333             0.550000  0.628571 0.715777 0.925061
HistGradientBoosting          0.733333             0.687500  0.709677 0.652996 0.930657


## 5. Group-Aware Hyperparameter Tuning (StratifiedGroupKFold)

Logistic Regression demonstrates the highest PR-AUC and perfect recall (100%) on the holdout.
We tune its regularization parameter $C$ on the training set using 5-fold `StratifiedGroupKFold` grouped by `batch_id` to maximize cross-validated Average Precision (PR-AUC).


In [12]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'model__C': [0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
    'model__solver': ['lbfgs']
}

grid_lr = GridSearchCV(
    Pipeline([('prep', prep), ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))]),
    param_grid,
    cv=sgkf.split(X_train, y_train, groups=groups_train),
    scoring='average_precision',
    n_jobs=1
)

grid_lr.fit(X_train, y_train)
print(f"Optimal C Parameter: {grid_lr.best_params_['model__C']}")
print(f"Best Group-Aware CV PR-AUC: {grid_lr.best_score_:.3f}")

best_model = grid_lr.best_estimator_
tuned_preds = best_model.predict(X_test)
tuned_probs = best_model.predict_proba(X_test)[:, 1]

print("")
print("Final Test Classification Report (Tuned Logistic Regression):")
print(classification_report(y_test, tuned_preds, target_names=['Safe', 'At Risk']))


Optimal C Parameter: 0.5
Best Group-Aware CV PR-AUC: 0.873

Final Test Classification Report (Tuned Logistic Regression):
              precision    recall  f1-score   support

        Safe       1.00      0.83      0.91       137
     At Risk       0.57      1.00      0.72        30

    accuracy                           0.86       167
   macro avg       0.78      0.92      0.82       167
weighted avg       0.92      0.86      0.88       167



## 6. Evaluation Curves & Confusion Matrix

We inspect:
1. Confusion Matrix on the holdout test set.
2. Precision-Recall Curve (benchmarking against baseline no-skill rate).
3. ROC Curve.
4. Predicted Probability distribution by true class.


In [14]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, tuned_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Safe', 'At Risk'], yticklabels=['Safe', 'At Risk'])
axes[0, 0].set_title(f"Confusion Matrix (Holdout Test) - Recall on At-Risk: {cm[1,1]}/{cm[1,0]+cm[1,1]} ({cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%)")
axes[0, 0].set_xlabel('Predicted Class')
axes[0, 0].set_ylabel('True Class')

# 2. Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, tuned_probs)
pr_auc = average_precision_score(y_test, tuned_probs)
axes[0, 1].plot(rec, prec, color='#1f77b4', lw=2, label=f'Tuned Model (PR-AUC = {pr_auc:.3f})')
axes[0, 1].axhline(y_test.mean(), color='red', linestyle='--', label=f'Baseline Rate ({y_test.mean()*100:.1f}%)')
axes[0, 1].set_title('Precision-Recall Curve (Holdout Test)')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].legend()

# 3. ROC Curve
fpr, tpr, _ = roc_curve(y_test, tuned_probs)
roc_auc = roc_auc_score(y_test, tuned_probs)
axes[1, 0].plot(fpr, tpr, color='#2ca02c', lw=2, label=f'Tuned Model (ROC-AUC = {roc_auc:.3f})')
axes[1, 0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1, 0].set_title('ROC Curve (Holdout Test)')
axes[1, 0].set_xlabel('False Positive Rate')
axes[1, 0].set_ylabel('True Positive Rate')
axes[1, 0].legend()

# 4. Predicted Probability Distribution
axes[1, 1].hist(tuned_probs[y_test == 0], bins=20, alpha=0.6, color='#1f77b4', label='Safe Batches', density=True)
axes[1, 1].hist(tuned_probs[y_test == 1], bins=20, alpha=0.6, color='#d62728', label='At-Risk Batches', density=True)
axes[1, 1].set_title('Predicted Probability Distribution by Ground Truth')
axes[1, 1].set_xlabel('Predicted Probability of Expiry Risk')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../models/expiry_risk_diagnostics.png', dpi=150)
plt.close()


## 7. Model Explainability & Odds Ratios

We interpret the model using:
1. **Standardized Coefficients & Odds Ratios**: Directly quantifies the multiplicative change in the odds of expiry per standard deviation change in feature value.
2. **Permutation Importance**: Measures degradation in PR-AUC when each feature is shuffled.


In [16]:
cat_names = best_model.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
all_names = cat_names + num_cols
coefs = best_model.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'Feature': all_names,
    'Coefficient': coefs,
    'Odds Ratio': np.exp(coefs)
}).sort_values('Coefficient', ascending=False).reset_index(drop=True)

print("Top 5 Risk-Increasing Factors (Higher Odds of Expiry):")
print(coef_df.head(5).to_string(index=False))

print("")
print("Top 5 Risk-Decreasing Protective Factors (Lower Odds of Expiry):")
print(coef_df.tail(5).to_string(index=False))

# Permutation importance on Test Set
perm = permutation_importance(best_model, X_test, y_test, scoring='average_precision', n_repeats=5, random_state=42, n_jobs=1)
perm_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance (PR-AUC Drop)': perm.importances_mean
}).sort_values('Importance (PR-AUC Drop)', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 6))
sns.barplot(data=perm_df.head(10), x='Importance (PR-AUC Drop)', y='Feature', hue='Feature', palette='Reds_r', legend=False)
plt.title('Top 10 Feature Importances (Permutation PR-AUC Drop on Holdout Test)')
plt.xlabel('Mean Drop in PR-AUC when Shuffled')
plt.tight_layout()
plt.savefig('../models/expiry_risk_feature_importance.png', dpi=150)
plt.close()

print("")
print("Top 10 Features by Permutation Importance:")
print(perm_df.head(10).to_string(index=False))


Top 5 Risk-Increasing Factors (Higher Odds of Expiry):
            Feature  Coefficient  Odds Ratio
     coverage_ratio     1.420045    4.137306
     store_id_ST_02     0.746855    2.110352
         unit_price     0.606267    1.833575
          unit_cost     0.588134    1.800624
category_Antibiotic     0.490744    1.633531

Top 5 Risk-Decreasing Protective Factors (Lower Odds of Expiry):
                   Feature  Coefficient  Odds Ratio
  est_demand_before_expiry    -0.447882    0.638980
       hist_avg_demand_30d    -0.519395    0.594880
            days_to_expiry    -0.589347    0.554690
           shelf_life_days    -0.598454    0.549660
remaining_shelf_life_ratio    -0.976958    0.376455

Top 10 Features by Permutation Importance:
                   Feature  Importance (PR-AUC Drop)
            coverage_ratio                  0.245991
remaining_shelf_life_ratio                  0.122055
                  store_id                  0.027325
       hist_avg_demand_30d               

## 8. Model Serialization

We save the trained champion pipeline to `models/expiry_risk_model.joblib`.


In [18]:
joblib.dump(best_model, '../models/expiry_risk_model.joblib')
print("Expiry-risk pipeline successfully serialized to models/expiry_risk_model.joblib")


Expiry-risk pipeline successfully serialized to models/expiry_risk_model.joblib
